In [1]:
import pandas as pd
import hashlib
import requests
import matplotlib.pyplot as plt



In [2]:
# Configuration
sampled_rows = []
total_kept = 0
target = 100000

# Stream dataset from Hugging Face
print("Connecting to Hugging Face...")
print("Getting Parquet file URLs...")
response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": "electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset"}
)

parquet_files = response.json()["parquet_files"]

print(f"Total Parquet files: {len(parquet_files)}")
print(f"These files together contain the FULL 5,000,000 rows\n")


for file_info in parquet_files:
    print(f"Loading {file_info['url'].split('/')[-1]}...")
    
    # Load one Parquet file
    df = pd.read_parquet(file_info["url"])
    print(f"  Loaded {len(df):,} rows")
    
    # Check each row
    for idx, row in df.iterrows():
        # Create unique string
        unique_string = f"{row['transaction_id']}_{row['is_fraud']}_{row['ip_geo_region']}"
        
        # Compute hash (0-99)
        hash_value = int(hashlib.md5(unique_string.encode()).hexdigest(), 16) % 100
        
        # Keep if hash is 0 or 1 (2% of rows)
        if hash_value < 2:
            sampled_rows.append(row.to_dict())
            total_kept = total_kept + 1
            
            # Stop when we have enough
            if total_kept >= target:
                break
    
    print(f"  Kept {total_kept} rows so far")
    
    # Stop if we have enough
    if total_kept >= target:
        print("Reached target! Stopping.")
        break

print(f"\n Done! Collected {len(sampled_rows)} rows")

transactions_data_sample = pd.DataFrame(sampled_rows)
transactions_data_sample.to_csv('transactions_sample_100k.csv', index=False)
print(f"Saved {len(transactions_data_sample):,} rows to 'transactions_sample_100k.csv'")

print(f"Sample size: {len(transactions_data_sample):,}")
print(f"Fraud rate: {transactions_data_sample['is_fraud'].mean():.2%}")
print("\nFirst 5 rows:")
print(transactions_data_sample.head())


Connecting to Hugging Face...
Getting Parquet file URLs...
Total Parquet files: 5
These files together contain the FULL 5,000,000 rows

Loading 0000.parquet...
  Loaded 1,220,000 rows
  Kept 24372 rows so far
Loading 0001.parquet...
  Loaded 1,220,000 rows
  Kept 48881 rows so far
Loading 0002.parquet...
  Loaded 1,220,000 rows
  Kept 73183 rows so far
Loading 0003.parquet...
  Loaded 1,220,000 rows
  Kept 97422 rows so far
Loading 0004.parquet...
  Loaded 120,000 rows
  Kept 99797 rows so far

 Done! Collected 99797 rows
Saved 99,797 rows to 'transactions_sample_100k.csv'
Sample size: 99,797
Fraud rate: 3.50%

First 5 rows:
  transaction_id                   timestamp  sender_account  \
0       T2328355  2023-11-27 01:35:41.098319      1000166601   
1       T1895250  2023-12-04 00:28:15.253259      1000255193   
2       T3479881  2023-11-26 03:33:02.469169      1000354808   
3        T978830  2023-05-26 03:49:28.946198      1000443816   
4       T2886206  2023-03-11 09:30:50.721998   